# EKKO: purchasing document header (EDA)

One row per purchasing document (PO, RFQ, contract, ...). Gives us the document category, supplier,
organisational scope (company code, purchasing org) and the document date used for the monthly trend.

Raw export: 3,803 rows x 195 columns. Most columns are empty or constant, so `sap_data.load` keeps only
the 12 we need (see the appendix at the bottom for the check).

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # src/, for sap_data

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sap_data import load, load_raw, PURCH_ORG, COMPANY_CODE, PERIOD_START, PERIOD_END

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [ ]:
ekko = load("ekko")
print(ekko.shape)
display(ekko.head())
ekko.dtypes

## 1. Grain
Expected: one row per `Purchasing Document`.

In [ ]:
assert ekko["Purchasing Document"].is_unique
ekko["Purchasing Document"].astype(str).str[:2].value_counts()

> **Note.** Grain confirmed: one row per document. Number ranges follow the document type
> (45… purchase orders, 60… RFQs, 46… contracts, 70…/80… requests).

## 2. Missing values

In [ ]:
ekko.isna().agg(["sum", "mean"]).T.sort_values("sum", ascending=False)

> **Note.** `Currency` is missing on ~8% of headers, `Document Date` on 4% (all of them category O
> requests, so no PO loses its date), and `Supplier` on ~3%. `Company Code`, `Purch. Organization` and
> `Purchasing Group` are missing on a few dozen. Sections 4–6 show where these gaps fall.

## 3. Document categories and types
`Purch. Doc. Category`: **F** = purchase order, **A** = RFQ/quotation, **K** = contract,
**O**/**R** = other request types. Only **F** carries ordered value ("Net PO value").

In [ ]:
pd.crosstab(ekko["Purch. Doc. Category"], ekko["Purchasing Doc. Type"], margins=True)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ekko["Purch. Doc. Category"].value_counts().plot.barh(ax=ax)
ax.set(title="EKKO documents by category", xlabel="Documents", ylabel="")
ax.invert_yaxis()

> **Note.** 1,850 of 3,803 headers (49%) are purchase orders (category F, type NB + 1 DB).
> 1,676 are RFQs (A/AN), 63 contracts (K/MK) and 214 other requests (O/R).
> **Decision:** keep only category `F` for every Net PO value measure.

## 4. Organisational scope (POs only)

In [ ]:
f = ekko[ekko["Purch. Doc. Category"] == "F"]
pd.crosstab(f["Purch. Organization"].fillna("(missing)"),
            f["Company Code"].fillna("(missing)"), margins=True)

> **Note.** Nearly all POs are US00 / US00. A handful sit in other orgs (`BIO`, `DE00`, `GL00`), and
> ~48 POs have no purchasing org at all; some of those still carry company code US00.
> **Decision:** US scope = `Purch. Organization == "US00"` (as the assignment suggests). The
> `scope_eda` notebook checks how much value the company-code fallback would add (it is negligible).

## 5. Currency (POs only)

In [ ]:
display(f["Currency"].value_counts(dropna=False))
f[f["Currency"].ne("USD")][["Purchasing Document", "Document Date", "Currency", "Exchange Rate",
                             "Company Code", "Purch. Organization", "Supplier"]]

> **Note.** 1,812 of 1,850 POs are USD. 3 EUR and 2 HUF POs exist, all dated before Oct 2025 and/or
> outside purch. org US00, so the reporting window is not affected. ~33 POs have no currency; most also
> lack company code and purch. org (incomplete exercise POs).
> **Decision:** report in USD. In-scope POs with a blank currency belong to company code US00, whose
> local currency is USD, so they are treated as USD (verified in `scope_eda`).

## 6. Supplier on the header

In [ ]:
ekko.groupby("Purch. Doc. Category")["Supplier"].agg(
    docs="size", missing=lambda s: s.isna().sum(), distinct="nunique")

> **Note.** 55 POs (3%) have no supplier on the header; all R-category requests are supplier-less by
> design. Those PO items will show up as "unknown supplier" after the join.

## 7. Dates

In [ ]:
display(ekko[["Document Date", "Created On"]].describe())
lag = (ekko["Created On"] - ekko["Document Date"]).dt.days
print("Document Date != Created On:", (lag != 0).sum(), "docs")
ekko[ekko["Document Date"] > pd.Timestamp.today()][
    ["Purchasing Document", "Purch. Doc. Category", "Document Date", "Created On"]]

In [ ]:
monthly = (f.groupby(f["Document Date"].dt.to_period("M")).size()
             .reindex(pd.period_range("2024-09", "2026-12", freq="M"), fill_value=0))
fig, ax = plt.subplots(figsize=(11, 3.5))
monthly.plot.bar(ax=ax, width=0.8)
ax.axvspan(monthly.index.get_loc(pd.Period(PERIOD_START, "M")) - 0.5,
           monthly.index.get_loc(pd.Period(PERIOD_END, "M")) - 0.5, color="C1", alpha=0.12)
ax.set(title="Purchase orders (category F) per month by Document Date; shaded = reporting window",
       xlabel="", ylabel="POs")
plt.xticks(rotation=90);

> **Note.** Documents run from Aug 2021 (one GBI demo PO) to Oct 2026. Activity is bursty: Sep-Dec 2024,
> Mar-May 2025, Oct-Dec 2025, Apr-May 2026, Oct 2026. Many months have zero POs. These bursts follow
> the teaching terms in the shared SAP client, not business seasonality.
> - Oct 2026 is the current, incomplete month, so the window of the last 12 complete months is
>   **1 Oct 2025 to 30 Sep 2026**.
> - One PO (4500001465) has a Document Date in Dec 2026, after its creation date. It falls outside the
>   window anyway.
> - `Document Date` differs from `Created On` for ~170 docs (back/forward-dated by users). We use
>   `Document Date` as the PO date.

## 8. Who created the documents

In [ ]:
ekko["Created By"].str.extract(r"^([A-Z]+)")[0].value_counts().head(10)

> **Note.** Creators are training users (`GBIALL…`, `LEARN-…`, `TEACH-…` plus individual student IDs).
> This is a shared SAP training client. It's a data-quality caveat for the report: volumes reflect class
> exercises, not real purchasing behaviour.

## Summary for the report (EKKO)
- Grain: 1 row per purchasing document; 3,803 docs, of which **1,850 are POs (category F)**.
- Filters used later: `Purch. Doc. Category == "F"`, `Purch. Organization == "US00"`.
- Currency: USD (non-USD POs fall outside the scope; blank currency is treated as USD for US00).
- Period: `Document Date` in [2025-10-01, 2026-10-01). Monthly activity is bursty and has empty months.

## Appendix: columns left out
Checks that the 183 columns we skip are empty or constant (reads the full Excel file, so it's slow).

In [ ]:
raw = load_raw("ekko")
informative = [c for c in raw.columns if raw[c].nunique(dropna=True) > 1]
print(f"{raw.shape[1]} columns, {len(informative)} with more than one distinct value:")
print(informative)